# Random Forest

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, train_test_split
import pickle

### Definir get_metrics

In [2]:
def get_classifier_metrics(y_predict_test, y_test, y_predict_train, y_train, average='micro'):
    metrics_train = (accuracy_score(y_train, y_predict_train),
                     f1_score(y_train, y_predict_train, average=average),
                     precision_score(y_train, y_predict_train, average=average),
                     recall_score(y_train, y_predict_train, average=average))
    metrics_test = (accuracy_score(y_test, y_predict_test),
                    f1_score(y_test, y_predict_test, average=average),
                    precision_score(y_test, y_predict_test, average=average),
                    recall_score(y_test, y_predict_test, average=average))
    return pd.DataFrame(data=[metrics_train, metrics_test],
                        columns=['Accuracy', 'F1 Score', 'Precision', 'Recall'],
                        index=['Train set', 'Test set'])

### Recopliacion de datos

In [3]:
df= pd.read_csv("../data/processed/diabetes_imputado.csv")
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6.0,148.0,72.0,35.0,0.0,33.6,0.6270,50.0,1.0
1,1.0,85.0,66.0,29.0,0.0,26.6,0.3510,31.0,0.0
2,8.0,183.0,64.0,0.0,0.0,23.3,0.6720,32.0,1.0
3,1.0,89.0,66.0,23.0,94.0,28.1,0.1670,21.0,0.0
4,0.0,137.0,40.0,35.0,168.0,43.1,0.7048,33.0,1.0


#### Split

In [4]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=42)

### Modelado

In [5]:
model= RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
y_predict_train = model.predict(X_train)
y_predict_test = model.predict(X_test)
metrics = get_classifier_metrics(y_predict_test, y_test, y_predict_train, y_train)
metrics

,Accuracy,F1 Score,Precision,Recall
Train set,1.000000,1.000000,1.000000,1.000000
Test set,0.772727,0.772727,0.772727,0.772727


### Optimización del modelo

In [6]:
hiperparameters = {
    'n_estimators': [10, 50, 100],
    'max_depth': [None, 10, 20],
    'bootstrap': [True, False],
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_features': ['auto', 'sqrt', 'log2'],
    'class_weight': [None, 'balanced']
}
grid_search = GridSearchCV(estimator=model,
                           param_grid=hiperparameters,
                           scoring='accuracy',
                           cv=5,
                           verbose=1,
                            n_jobs=-1)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
y_predict_train_best = best_model.predict(X_train)
y_predict_test_best = best_model.predict(X_test)
metrics_best = get_classifier_metrics(y_predict_test_best, y_test, y_predict_train_best, y_train)
metrics_best    

Fitting 5 folds for each of 324 candidates, totalling 1620 fits


/home/vscode/.local/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
540 fits failed out of a total of 1620.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
540 fits failed with the following error:
Traceback (most recent call last):
  File "/home/vscode/.local/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/vscode/.local/lib/python3.11/site-packages/sklearn/base.py", line 1356, in wrapper
    estimator._validate_params()
  File "/home/vscode/.local/lib/python3.11/site-packages/sklearn/base.py", line 469, in _validate_params
    validate_parameter_constraints(
  File "/home/vscode/.l

,Accuracy,F1 Score,Precision,Recall
Train set,1.00000,1.00000,1.00000,1.00000
Test set,0.74026,0.74026,0.74026,0.74026


### Guardado del modelo

In [7]:
with open('/workspaces/infoasir20-primer-algoritmo/models/random-forest-model.pkl', 'wb') as file:
    pickle.dump(model, file)    